# Notebook consacré à l'analyse des variables textuelles ouvertes

Exemple :




In [1]:

!python -m spacy download fr_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/45.8 MB ? eta -:--:--

     ━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/45.8 MB 71.3 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 27.5/45.8 MB 73.4 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 45.6/45.8 MB 83.4 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 MB 56.8 MB/s  0:00:00


✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_md')


In [2]:
import pandas as pd
import numpy as np
import spacy
import re

import nltk
from nltk.corpus import stopwords

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [3]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [5]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-07-24.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../../le_questionnaire/dico_variable.csv", sep = ",")
df0.columns

Index(['Unnamed: 0', 'qno_obs', 'q1_so_principles', 'q2_hal_depot',
       'q3_octavi_depot', 'q4_diff_data', 'q4_autres_entrepots', 'q5_r',
       'q5_python', 'q5_excel',
       ...
       'q16_ocr_help', 'q16_network_analysis_help', 'q16_odette_atom_help',
       'q16_caqdas_help', 'q16_carto_help', 'q16_sig_help', 'q16_sgbd_help',
       'q16_autre_help', 'q16_1_other_software_help', 'q26_1_autre_rec'],
      dtype='object', length=168)

In [ ]:
df_col["name"] = df_col.name.apply(lambda row : row.replace(' :', ''))

In [6]:

list_text = [x for x in df_col.label.loc[(df_col.type=="texte_libre")&(df_col.label!="q12_1_how_humanum")&(df_col.personal_data==False)]]



dict_col = {}
for col in list_text:
    dict_col[col] = len(df0.loc[~df0[col].isna()])
print(dict_col)

{'q4_autres_entrepots': 26, 'q5_1_other_lang': 37, 'q6_1_dev_exemple': 32, 'q7_1_other_sw_depot': 6, 'q9_utilite_accomp': 50, 'q10_other_type_data': 4, 'q17_util_accomp_software': 46, 'q18_1_utilite_accomp_methode': 46, 'q19_1_other_software_help': 4, 'q20_1_other_closed_software_help': 4, 'q21_util_accomp_closed_software': 46, 'q23_other_service_univ': 18, 'q25_1_statut_autre': 6, 'q26_1_autre_env': 27, 'q28_1_autre_archive_data': 15, 'q30_1_autre_webbrowser': 8, 'q31_1_autre_stockage': 13, 'q32_1_autre_collab_tools': 64, 'q33_1_autre_suavegarde': 12, 'q35_ia_tools': 68, 'q37_raison_util_ia': 68, 'q44_ufr_labo': 59, 'q16_1_other_software_help': 10}


In [7]:
nb_rep_question_ouverte = pd.DataFrame.from_dict(data = dict_col, orient='index')
nb_rep_question_ouverte

,0
q4_autres_entrepots,26
q5_1_other_lang,37
q6_1_dev_exemple,32
q7_1_other_sw_depot,6
q9_utilite_accomp,50
q10_other_type_data,4
q17_util_accomp_software,46
q18_1_utilite_accomp_methode,46
q19_1_other_software_help,4
q20_1_other_closed_software_help,4


In [8]:
nlp = spacy.load('fr_core_news_md')
nlp.add_pipe("merge_entities")
#nlp.add_pipe("merge_noun_chunks")
spacy_stopwords = list(nlp.Defaults.stop_words)




nltk.download('stopwords')

spacy_stopwords
stop_words = set(stopwords.words('french'))


[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# Recupération des affiliations

In [ ]:
df0[["q45_clé","q44_ufr_labo"]].loc[~df0.q44_ufr_labo.isna()].to_csv("list_affiliation.csv", sep =",", index = False)

In [9]:

for n, x in enumerate(df0.q44_ufr_labo.loc[~df0.q44_ufr_labo.isna()]):
    print(n+1, x)

1 UFR Psychologie
2 SDL, SFL
3 UFR de Psychologie, UR DysCo
4 UFR DROIT 
5 UFR Culture et communication / CEMTI
6 UFR de psychologie, Laboratoire DysCo
7 Psychologie, Laboratoire CHArt
8 IED / Laboratoire Paragraphe
9 UFR LLCE LEA 
10 Mitsic, Humanités numériques, Paragraphe
11 IDHES (UMR 8533)
12 UFR AES-EG, laboratoire LED
13 Textes et sociétés, Cresppa
14 UFR Arts, philosophie, esthétique, département Arts, AIAC/EPHA
15 SEPF
16 Psychologie
17 UFR Psychologie
18 IEE/cresppa
19 UFR Arts, Département Photo, Aiac, epha
20 Arts, Arts plastiques, AIAC
21 AES-EG, LED
22 UFR Psychologie, affiliation laboratoire : LAPPS
23 ED-CLI
24 UFR Psychologie, UR Paragraphe
25 UFR Psychologie, laboratoire DysCo
26 UFR Arts, Département Musique, Laboratoire Musidanse
27 UFR de psychologie
28 Laboratoire d'économie dionysien (LED)
29 UFR AES Economie Gestion / LED
30 UFR ERITES ; Géographie ; Ladyss
31 UFR LLCE/LEA
32 UFR Psychologie - CHART
33 MITSIC / LIASD
34 UFR eriTES
35 UFR eritES, géographie, UMR 

# Traitement texte long

In [10]:
df_test = df0[["q6_1_dev_exemple"]].loc[~df0["q6_1_dev_exemple"].isna()].copy()

    

In [11]:
def tokenizer(data, column, list_stopword = None, pos=['NOUN','VERB','ADJ','PROPN']):
    dict_token = {}
    df_test = data[[column]].loc[~data[column].isna()].copy()
    for row in df_test[column]:
        new_row = re.sub("·",".", row)
        doc = nlp(new_row)
        if len(doc) > 2:
            cleaned_doc = "|".join([t.text.lower() for t in doc if t.pos_ in pos])
        
        else:
            cleaned_doc = " ".join([t.text.lower() for t in doc])
    #chunk_l = [chunk.text.lower() for chunk in doc.noun_chunks]
        dict_token[row] = cleaned_doc
    return dict_token

In [13]:
tokenizer(df0, 'q37_raison_util_ia', list_stopword = None, pos=['NOUN'])

{'Chat GPT, Mistral IA, LM Notebook': 'chat gpt',
 'chatgpt, euria': 'chatgpt|euria',
 'Chat-GPT, Perplexity': '',
 'le chat ; chatgpt': 'chat|chatgpt',
 "Intelligence Artificielle désigne de nombreuses technologies. Je n'utilise pas d'outils génératifs mais je travaille avec et développe des outils de transcription, d'analyse de texte et de la parole, d'analyse d'image. Il s'agit de logiciels qui tournent en local sur mon matériel.": 'technologies|outils|outils|transcription|analyse|texte|parole|analyse|image|logiciels|local|matériel',
 'hugging face, openAI, gemini, midjourney': 'face|openai',
 'Cursor': 'cursor',
 'whisper sur sharedocs': 'sharedocs',
 'Lumo': 'lumo',
 'Gemini, Ollama, Copilot': '',
 "DeepL - objection de conscience par rapport à l'IA générative": 'objection|conscience|rapport',
 'gemini, chatpgt, claude': 'gemini',
 'chatgpt': 'chatgpt',
 'Claude, ChatPGT': '',
 'chatGTP, Perplexity, Emmy': '',
 "NoScribe en local pour des transcriptions d'entretiens + deepl pour d

In [ ]:
df_clean = tokenizer(df0, column = "q9_utilite_accomp", list_stopword = stop_words)
df_clean#[["cleaned_text", "tokens"]]

In [ ]:
other_sw_depot_rec = 'Sur les dépôts SSC des commandes Stata': 'archives du he Boston College Statistical Software Components',
 'sur des repositories personnels': 'dépôts personnels',
 'HAL': 'hal',
 'ORTOLANG / HUMA-NUM': 'ortolang|huma-num',
 'GitLab mais instance auto-hébergée': 'gitlab|instance|auto-hébergée',
 'APP / INPI': 'app|inpi'}

In [ ]:
df0["q4_autres_entrepot_rec"]= df0.q4_autres_entrepots.map(other_repo_rec.get)
df0

with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
    df0.to_csv(file_out, sep =",", index =False)



In [ ]:
df0

In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-05-11.csv", "w") as file_out:
    df0.rename(columns={"q4_autres_entrepot_rec":"q4_autres_entrepots_rec"}).to_csv(file_out, index =False, sep=",")

In [ ]:
q4 = df_col.loc[df_col.label=="q4_autres_entrepots"]
dict_col = {}
for x in q4.columns:
    dict_col[x] = q4[x].values[0]


In [ ]:
dict_col

In [ ]:
dic_new_var = {'name': '5.rec Autre, précisez',
 'label': "q4_autres_entrepots_rec",
 'group': '3_entrepot',
 'personal_data': False,
 'type': 'nominal_multiple',
 'opened_question': False,
 'type_panda': 'object',
 'no_question': 4.0,
 'question': '4/ Avez-vous déjà diffusé vos données de recherche une fois le projet terminé ?'
              }
new_variable = pd.DataFrame.from_dict(data=[dic_new_var])

new_variable
df_col = pd.concat([df_col,new_variable], ignore_index = True)
df_col.to_csv("../le_questionnaire/dico_variable.csv", sep = ",", index = False)

In [ ]:
#df_col = df_col.iloc[0:-2]#.to_csv("../le_questionnaire/dico_variable.csv", sep = ",", index = False)
df_col

In [ ]:
df_col.to_csv("../le_questionnaire/dico_variable.csv", sep = ",", index = False)

## traitement texte court

In [ ]:
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: re.sub(r",", "", str(row.q5_1_other_lang)).lower(), 1) # remove urls
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: re.sub(r"\bet\b", "", str(row.q5_1_other_lang_clean)).lower(), 1) # remove urls
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: "".join(str(row.q5_1_other_lang_clean).split(". ")[0]), 1) # remove urls
df0["tokens"] = df0.apply(lambda row: row.q5_1_other_lang_clean.split(),1) 
df0[["q5_1_other_lang_clean","tokens"]]

In [ ]:
for n, x in enumerate(df0.q5_1_other_lang_clean.loc[~df0.q5_1_other_lang.isna()]):
    print(n+1, x)

# Types de données

In [ ]:
split_multiple_choices(df0, column='q33_sauvegarde_data', index = "q45_clé")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(8, figsize=(10, 15))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple):
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total", y=col, data=gb_data,
                label="Total", color="b", ax=ax[n])
    sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])
    

sns.despine(left=True, bottom=True)
plt.savefig("multiple_choice.png")

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])